# Custom models

Define a callable Julia model, register its constructor, and use the registered constructor from a YAML config file.


In [ ]:
function find_repo_root(start = pwd())
    dir = abspath(start)
    while !isfile(joinpath(dir, "Project.toml")) || !isdir(joinpath(dir, "src"))
        parent = dirname(dir)
        parent == dir && error("Could not find QuantumGraph.jl repository root")
        dir = parent
    end
    dir
end

repo_root = find_repo_root()
cd(repo_root)
using Pkg
Pkg.activate(repo_root)


In [ ]:
using QuantumGraph
using DataFrames
using Flux
using Optimisers

struct ResidualMLP
    input::Flux.Dense
    hidden::Flux.Chain
    output::Flux.Dense
end
ResidualMLP(width::Integer) = ResidualMLP(Flux.Dense(2 => width, Flux.relu), Flux.Chain(Flux.Dense(width => width, Flux.relu), Flux.Dense(width => width)), Flux.Dense(width => 1))
function (model::ResidualMLP)(x::AbstractMatrix)
    h = model.input(x)
    model.output(h .+ model.hidden(h))
end
Flux.@layer ResidualMLP


In [ ]:
QuantumGraph.register_object!("Docs.ResidualMLP", kwargs -> ResidualMLP(Int(kwargs["width"])))
config_path = joinpath(repo_root, "docs", "src", "examples", "configs", "custom_model.yml")
file_config = load_config(read(config_path, String))
constructor = resolve_registered_object(file_config["model_constructor"].identifier)
model = constructor(Dict("width" => 4))
model(Float32[1 2; 3 4])


In [ ]:
batches = [(Float32[1 2 3 4; 2 3 4 5], Float32[3 5 7 9])]
loss(model, batch) = sum(abs2, vec(model(batch[1])) .- batch[2])
evaluator(model, iterator) = DataFrame(loss_avg = [loss(model, first(iterator))])
trainer = construct_trainer(Dict{String, Any}(
    "dataset" => batches,
    "model" => model,
    "optimizer" => Optimisers.Adam(0.01),
    "loss" => loss,
    "evaluator" => evaluator,
    "early_stopping" => early_stopping_state(metric = :loss_avg),
    "output_path" => mktempdir(),
    "device" => "cpu",
    "num_epochs" => 2,
))
fit_trainer!(trainer)
last(trainer.reports)
